## NPTEL Study Notes: Introduction to Large Language Models (LLMs)

**Topic: Advanced Smoothing Techniques and Language Model Evaluation**

These notes cover advanced statistical language modeling, moving beyond basic N-gram counts to handle zero-probability cases, and exploring how to evaluate language models mathematically and practically.

*(Note: Audio transcription errors from the lecture, such as "advanced smoothing" for Add-One smoothing, "blue score" for BLEU score, and "missing translation" for machine translation, have been corrected for technical accuracy.)*

---

## 1. Recap: Bigram Models & Add-One (Laplace) Smoothing

In a standard bigram model, the probability of a word given the previous word is based on simple frequency counts:


$$P(w_i \vert{} w_{i-1}) = \frac{C(w_{i-1}, w_i)}{C(w_{i-1})}$$

**Add-One (Laplace) Smoothing** attempts to fix the "zero probability" problem (unseen bigrams) by blindly adding 1 to every bigram count. To ensure probabilities still sum to 1, the vocabulary size $\vert{}V\vert{}$ is added to the denominator.

* **The Major Flaw:** It steals too much probability mass from highly frequent words and distributes it to unseen words. High-frequency counts drop drastically (e.g., 600 becomes 300), skewing the actual distribution.

---

## 2. Advanced Smoothing: Good-Turing Estimate

**Intuition:** Use the count of things we have seen *once* to estimate the probability of things we have *never* seen.

**Key Concept: Frequency of Frequencies ($N_c$)**
$N_c$ represents the number of items that occur exactly $c$ times in the corpus.

* $N_1$: Number of words/bigrams that appear exactly 1 time.
* $N_2$: Number of words/bigrams that appear exactly 2 times, etc.

**The Formula:**
Instead of using the raw count $c$, Good-Turing assigns a **modified count ($c^*$)**:


$$c^* = (c + 1) \frac{N_{c+1}}{N_c}$$

**Estimating Unseen Events (Zero Counts):**
For items with a count of $c=0$, the probability is estimated using $N_1$ (things seen exactly once) divided by the total number of observations $N$:


$$P_{GT}(\text{unseen}) = \frac{N_1}{N}$$

> **Worked Example: Bird Watching**
> You observe 18 birds total ($N = 18$). You see 10 flamingos, 2 kingfishers, 1 Indian roller, 1 woodpecker, and 1 peacock.
> * $N_1 = 3$ (roller, woodpecker, peacock all appear once).
> * $N_2 = 1$ (kingfisher appears twice).
> * **Maximum Likelihood Estimate (MLE)** for an unseen "purple heron" is $0/18 = 0$.
> * **Good-Turing Estimate** for an unseen "purple heron": $N_1 / N = 3/18$.
> * **Modified probability for woodpecker ($c=1$)**:
> $c^* = (1+1) \times (N_2 / N_1) = 2 \times (1/3) = 0.66$
> $P(\text{woodpecker}) = 0.66 / 18 = 1/27$. (Notice this is lower than the original $1/18$ MLE, as probability mass was redistributed to unseen events).
> 
> 

**Observation:** In practice, the Good-Turing modified count $c^*$ roughly equals $c - 0.75$. This observation leads directly to the next technique.

---

## 3. Absolute Discounting & Kneser-Ney Smoothing

### Absolute Discounting Interpolation

Since Good-Turing roughly subtracts a constant value from non-zero counts, **Absolute Discounting** formalized this by subtracting a fixed discount $d$ (often $\approx 0.75$) from all non-zero bigram counts, and interpolating the "leftover" probability mass with the unigram probability.

$$P(w_i \vert{} w_{i-1}) = \frac{\max(C(w_{i-1}, w_i) - d, 0)}{C(w_{i-1})} + \lambda(w_{i-1}) P(w_i)$$

### Kneser-Ney Smoothing

**The Flaw in Absolute Discounting:** Interpolating with standard unigram probability $P(w_i)$ is dangerous due to context-dependent high-frequency words.

* *Example:* In a corpus about the US, "Los Angeles" appears frequently. Therefore, "Angeles" has a very high unigram count. If a sentence reads "I am drinking a cup of ___", the model might incorrectly predict "Angeles" over "coffee" because $P(\text{Angeles}) > P(\text{coffee})$.

**The Solution: Continuation Probability**
Kneser-Ney replaces the unigram probability with **Continuation Probability**. Instead of asking "how often does word $w$ appear?", it asks "how many *unique bigrams* does word $w$ complete?".

* "Angeles" almost exclusively follows "Los" (completes 1 unique bigram).
* "Coffee" follows many words: "hot", "cold", "morning", "breakfast" (completes many unique bigrams).

**Continuation Probability Formula:**


$$P_{continuation}(w) = \frac{\vert{}\{w_{i-1} : C(w_{i-1}, w) > 0\}\vert{}}{\vert{}\{(w_{j-1}, w_j) : C(w_{j-1}, w_j) > 0\}\vert{}}$$

* **Numerator:** Number of unique preceding words for $w$.
* **Denominator:** Total number of unique bigrams in the corpus.

**Calculating $\lambda$ (The Interpolation Weight):**
To ensure probabilities sum to 1, the discounted mass $d$ must be redistributed.


$$\lambda(w_{i-1}) = \frac{d}{C(w_{i-1})} \times \vert{}\{w : C(w_{i-1}, w) > 0\}\vert{}$$


This essentially means: (Discount factor / Total row count) $\times$ (Number of non-zero entries in that specific row of the bigram matrix).

---

## 4. Evaluating Language Models

How do we know if Language Model A is better than Language Model B?

### 1. Extrinsic (Task-Based) Evaluation

Feed the language model's output into a downstream task (e.g., Machine Translation, Speech Recognition) and evaluate the task's metric (e.g., BLEU score for translation).

* **Drawbacks:**
1. Computationally expensive and time-consuming (can take days to run).
2. Hard to pick which downstream task to use as the benchmark.
3. The downstream model's own poor quality might mask the language model's actual performance.



### 2. Intrinsic Evaluation (Perplexity)

Evaluate the model independently based on how well it predicts a test dataset. The standard metric is **Perplexity (PP)**.

* **Definition:** The inverse probability of the test set, normalized by the number of words. It represents the geometric mean of the inverse probability.

$$PP(W) = P(w_1, w_2, \dots, w_n)^{-\frac{1}{n}} = \sqrt[n]{\frac{1}{P(w_1, w_2, \dots, w_n)}}$$


* **Rule of Thumb:** A **lower** perplexity indicates a **better** model. High probability of the test set = low perplexity.

---

## 5. Information Theory: Entropy & Perplexity

Perplexity is deeply connected to Shannon's Information Theory.

### Entropy ($H$)

Entropy measures randomness or uncertainty. Maximum entropy occurs in a uniform distribution (all outcomes equally likely).


$$H(X) = - \sum p(x) \log_2 p(x)$$


Intuitively, entropy represents the *expected minimum number of bits required to encode an information event*.

### Entropy Rate (Per-Word Entropy)

For a language (a sequence of tokens), we look at the entropy per word as the sequence length approaches infinity:


$$H(L) = \lim_{n \to \infty} \frac{1}{n} H(w_1, w_2, \dots, w_n)$$

**Shannon-McMillan-Breiman Theorem:** If a language is a regular stochastic process (stationary and ergodic), we can approximate the sum over all possible sentences by taking a single, significantly long sentence.


$$H(L) \approx \lim_{n \to \infty} - \frac{1}{n} \log_2 P(w_1, w_2, \dots, w_n)$$

### Cross-Entropy

We don't know the true distribution of a language ($L$); we only have our model ($M$). **Cross-Entropy** measures how well our model $M$ approximates the true language $L$.


$$H(L, M) = - \sum P_L(x) \log_2 P_M(x)$$

* **KL Divergence bound:** $H(L, M) = H(L) + D_{KL}(L \vert{}\vert{} M)$.
* Since KL Divergence is always $\ge 0$, the model's cross-entropy is always an upper bound to the true entropy: $H(L, M) \ge H(L)$.

### The Relationship: Perplexity = $2^{Entropy}$

$$PP = 2^H$$

* If entropy $H$ is the *number of bits* required to encode the next word, $2^H$ is the *number of possible words* those bits can represent.
* **Intuition:** Perplexity is the model's **weighted branching factor**. A perplexity of 100 means the model is, on average, as confused as if it had to guess uniformly among 100 possible next words.

---

## 6. Limitations of N-Gram Language Models

Despite smoothing, statistical N-gram models have fundamental limitations that pave the way for Neural Language Models:

1. **Fixed Context Window:** They only look 1 or 2 words back, missing long-range dependencies.
2. **Sparsity & Exponential Growth:** The bigram/trigram matrix grows exponentially with vocabulary size, resulting in mostly zero counts.
3. **Out of Vocabulary (OOV):** Extremely poor at handling newly coined words or words not seen in the training data.
4. **Computational Cost:** Storing and querying giant sparse matrices for trillion-word corpora is highly inefficient.

---

## Practice Questions

**Q1. What is the primary disadvantage of Add-One (Laplace) smoothing in statistical language modeling?**
A) It requires complex calculus to compute the modified probabilities.
B) It steals an excessive amount of probability mass from high-frequency bigrams and distributes it to zero-frequency bigrams.
C) It cannot assign non-zero probabilities to completely unseen words.
D) It relies heavily on unigram continuation probabilities.
**Answer:** **B**. Add-one smoothing blindly adds 1 to everything, which drastically reduces the counts of frequently occurring words (e.g., dropping a count of 600 to 300) to account for the massive number of zero-count occurrences in a sparse matrix.

**Q2. You are given a corpus where 40 words appear exactly once ($N_1 = 40$), 15 words appear exactly twice ($N_2 = 15$), and the total number of words observed is 1000. Using the Good-Turing estimate, what is the modified count ($c^*$) for a word that currently appears exactly once?**
*Step-by-step reasoning:*

1. The formula for the modified count is $c^* = (c + 1) \frac{N_{c+1}}{N_c}$.
2. The current count $c = 1$.
3. We need $N_1$ and $N_2$. $N_1 = 40$ and $N_2 = 15$.
4. Substitute values: $c^* = (1 + 1) \times \frac{15}{40}$.
5. $c^* = 2 \times 0.375 = 0.75$.
**Answer:** The modified count $c^*$ is **0.75**.

**Q3. Why does Kneser-Ney smoothing use "continuation probability" instead of standard unigram probability?**
A) To speed up the mathematical interpolation process during training.
B) To heavily penalize words that have low bigram frequencies.
C) To prevent words that appear frequently but only in one specific context (like "Angeles" after "Los") from being erroneously predicted in new, unrelated contexts.
D) To ensure that the sum of all probabilities in the bigram matrix equals the vocabulary size.
**Answer:** **C**. Continuation probability measures how many *unique* preceding words a token has. This prevents context-locked words (like "Angeles") from dominating predictions just because their raw frequency is high.

**Q4. If a language model has an entropy rate of 4 bits per word, what is its perplexity?**
*Step-by-step reasoning:*

1. The relationship between Perplexity (PP) and Entropy (H) is given by $PP = 2^H$.
2. The given entropy $H = 4$.
3. $PP = 2^4 = 16$.
**Answer:** The perplexity is **16**.

---

## Quick Revision Summary

* **Laplace Smoothing:** Adds 1 to all counts. Flawed because it penalizes rich/frequent counts too heavily.
* **Good-Turing:** Uses frequency of frequencies. Estimates unseen events using $N_1 / N$. Modifying a count essentially subtracts $\approx 0.75$ from it.
* **Absolute Discounting:** Subtracts fixed $d$ from non-zero counts, interpolates remainder with unigrams.
* **Kneser-Ney:** Upgrades absolute discounting by replacing unigrams with **continuation probability** (how many unique bigrams a word completes) to solve the "Los Angeles" problem.
* **Extrinsic Evaluation:** Testing on downstream tasks (MT, Speech). Slow, noisy.
* **Intrinsic Evaluation:** Testing via **Perplexity**. Lower is better.
* **Perplexity vs Entropy:** $PP = 2^H$. Perplexity acts as a branching factor—how many words the model is choosing between on average. Cross-entropy of the model is the upper bound of the true language entropy.